## Quantize tell-tale classifier (YOLO-cls) to FP16

Downloads the same **3-class** YOLO dataset zip as [finetuning/train_classifier.ipynb](../finetuning/train_classifier.ipynb), loads `sailcv-yolo11n-cls224.pt` from Hugging Face (same bucket as `model_weights.HF_WEIGHT_FILES`), converts to half precision, saves a new checkpoint.

**GPU recommended:** On Colab: **Runtime → Change runtime type → GPU**.

**Output:** e.g. `sailcv-yolo11n-cls224_fp16.pt`. Set `classifier.model_path` in your parameters YAML to this file (or copy to `checkpoints/`).

In [ ]:
!nvidia-smi

In [ ]:
import os

HOME = os.getcwd()
print(HOME)

## Install dependencies

In [ ]:
!pip install -q "ultralytics>=8.3.0" huggingface_hub opencv-python-headless numpy tqdm pyyaml
from IPython import display

display.clear_output()
print("OK")

## Download and unzip main dataset from Hugging Face

File: `telltale_3_classes_fused.zip` (YOLO layout: `train/`, `val/`, `data.yaml`, `nc: 3`). Same flow as the classifier finetuning notebook.

In [ ]:
import zipfile
from pathlib import Path

from huggingface_hub import hf_hub_download

main_data_dir = Path(HOME) / "main_data"
main_data_dir.mkdir(parents=True, exist_ok=True)

zip_path = hf_hub_download(
    repo_id="estefoucher/sail-cv-telltales",
    filename="telltale_3_classes_fused.zip",
    repo_type="dataset",
    local_dir=HOME,
    local_dir_use_symlinks=False,
)
with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(main_data_dir)

contents = [p for p in main_data_dir.iterdir() if p.is_dir()]
candidate = main_data_dir
for child in contents:
    if (child / "train").exists() or (child / "val").exists():
        candidate = child
        break
if not (candidate / "train").exists() and not (candidate / "val").exists():
    raise FileNotFoundError(
        f"Expected train/ or val/ under {main_data_dir}. Contents: {list(main_data_dir.iterdir())}"
    )
main_data_dir = candidate.resolve()
print(f"Main dataset at: {main_data_dir}")
for split in ("train", "val"):
    img_dir = main_data_dir / split / "images"
    n = len(list(img_dir.iterdir())) if img_dir.exists() else 0
    print(f"  {split}/images: {n} items")

## Download classifier checkpoint from Hugging Face

To use a **local** `best.pt` from your own training instead, set `ckpt_path` to that path and skip the download cell.

In [ ]:
from huggingface_hub import hf_hub_download

ckpt_path = hf_hub_download(
    repo_id="estefoucher/tell-tale-detector",
    filename="weights/sailcv-yolo11n-cls224.pt",
    local_dir=HOME,
    local_dir_use_symlinks=False,
)
print(f"Checkpoint: {ckpt_path}")

## Load, convert to FP16, save

Uses Ultralytics `YOLO`: move to device, `half()`, then `save()`.

In [ ]:
import torch
from pathlib import Path
from ultralytics import YOLO

OUT_NAME = "sailcv-yolo11n-cls224_fp16.pt"
out_path = Path(HOME) / OUT_NAME

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

model = YOLO(str(ckpt_path))
model.to(device)
model.half()
model.save(str(out_path))
print(f"Saved FP16 checkpoint: {out_path}")

## Optional: smoke test on val crops (YOLO boxes)

Reloads the saved FP16 checkpoint, samples val images, takes **one random bounding box** per image from YOLO labels, crops and resizes to 224×224 (classification input size), then runs the classifier—closer to pipeline crops than full-frame inference.

In [ ]:
import random
import time
from pathlib import Path

import cv2
import torch
import yaml
from tqdm.auto import tqdm
from ultralytics import YOLO

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".tif"}


def yolo_lines(lbl_path: Path) -> list[tuple[int, float, float, float, float]]:
    out = []
    if not lbl_path.is_file():
        return out
    for raw in lbl_path.read_text().splitlines():
        s = raw.strip()
        if not s:
            continue
        parts = s.split()
        if len(parts) < 5:
            continue
        cls = int(float(parts[0]))
        xc, yc, w, h = map(float, parts[1:5])
        out.append((cls, xc, yc, w, h))
    return out


def xywhn_to_xyxy_pixel(xc, yc, w, h, W, H):
    x1 = (xc - w / 2) * W
    y1 = (yc - h / 2) * H
    x2 = (xc + w / 2) * W
    y2 = (yc + h / 2) * H
    return int(x1), int(y1), int(x2), int(y2)


val_img = Path(main_data_dir) / "val" / "images"
val_lbl = Path(main_data_dir) / "val" / "labels"
images = [p for p in val_img.iterdir() if p.is_file() and p.suffix.lower() in IMG_EXTS]
if not images:
    print("No val images; skip crop smoke test.")
else:
    names = None
    yp = Path(main_data_dir) / "data.yaml"
    if yp.is_file():
        meta = yaml.safe_load(yp.read_text())
        raw = meta.get("names", {})
        if isinstance(raw, dict):
            keys = sorted(raw, key=lambda k: int(str(k)))
            names = [str(raw[k]) for k in keys]
        elif isinstance(raw, list):
            names = [str(x) for x in raw]
    sample = random.sample(images, k=min(32, len(images)))
    device = "cuda" if torch.cuda.is_available() else "cpu"
    m = YOLO(str(out_path))
    m.to(device)
    t0 = time.perf_counter()
    n_ok = 0
    for p in tqdm(sample, desc="FP16 cls crops"):
        im = cv2.imread(str(p))
        if im is None:
            continue
        H, W = im.shape[:2]
        stem = p.stem
        lines = yolo_lines(val_lbl / f"{stem}.txt")
        if not lines:
            continue
        _, xc, yc, bw, bh = random.choice(lines)
        x1, y1, x2, y2 = xywhn_to_xyxy_pixel(xc, yc, bw, bh, W, H)
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(W, x2), min(H, y2)
        if x2 <= x1 or y2 <= y1:
            continue
        crop = im[y1:y2, x1:x2]
        if crop.size == 0:
            continue
        crop224 = cv2.resize(crop, (224, 224), interpolation=cv2.INTER_LINEAR)
        r = m(crop224, imgsz=224, verbose=False)
        pr = r[0]
        if hasattr(pr, "probs") and pr.probs is not None:
            n_ok += 1
    dt = time.perf_counter() - t0
    print(f"Crops classified: {n_ok}/{len(sample)}, time: {dt:.2f}s")
    if names:
        print("Class names (order):", names)

## Output

Use `sailcv-yolo11n-cls224_fp16.pt` as `classifier.model_path` in your tracking parameters YAML (e.g. `parameters/default_classifier.yml`).